# Grupo N · Detección de lechugas: Cascade R-CNN y SSD300
**Visión por computador · 2026–2027**  
Defensa: lunes 30 de noviembre, 12:00 · Aula IA.14.

Notebook experimental, no informe con resultados precalculados. Ejecutar los entrenamientos para obtener las conclusiones. No se ha ejecutado aquí contra una GPU ni validado la instalación de Detectron2 en tu máquina.

## Objetivos y protocolo
1. Conservar las particiones oficiales de Francesco/lettuce-pallets.
2. Ajustar ambos modelos desde COCO; sustituir exclusivamente las salidas de clasificación incompatibles.
3. Comparar baseline, aumento geométrico/fotométrico, AdamW y tasas diferenciadas mediante ablaciones independientes (cada experimento parte de COCO).
4. Seleccionar por AP@[.50:.95] de validación, nunca de test. Comparar también TTA horizontal y ensemble con NMS por clase.
5. Bloquear decisiones antes del test; generar AP50, AP, AP75, AP por tamaño/clase, PR, latencia y ejemplos de aciertos/errores.

Las mejoras son hipótesis, no garantías. La vista cenital motiva simetrías; el color puede informar del crecimiento, por eso el jitter es moderado y no se altera el tono. No se cambia SSD300 a otra resolución. La comparación usa recetas con igual número de exposiciones aproximadas, no igual coste computacional: Cascade usa FPN/640 y SSD 300×300. No atribuir toda diferencia exclusivamente a one-stage/two-stage.

## Entorno
Recomendado: Linux, Python 3.10/3.11, GPU CUDA y espacio persistente. Detectron2 compila extensiones: instalarlo después de PyTorch y reiniciar el kernel si se cambian paquetes ya importados. No se reemplaza automáticamente tu versión de PyTorch/CUDA. En Colab seleccionar GPU primero. Si hay un error de compilación, revisar la compatibilidad de torch/torchvision/CUDA y el compilador antes de entrenar.


In [ ]:
# Ejecutar una vez en un entorno con torch y torchvision compatibles ya instalados.
%pip install -q "numpy<2" "pillow>=10,<12" "datasets==3.6.0" "huggingface_hub>=0.30,<1" pycocotools pandas matplotlib seaborn tqdm ninja fvcore iopath
%pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"


In [ ]:
import os, sys, json, time, random, math, hashlib, subprocess, gc, copy, tarfile
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.ops import batched_nms, box_iou
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead
from datasets import load_dataset
from huggingface_hub import hf_hub_download, HfApi
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.structures import Instances, Boxes
from detectron2.data.transforms import ResizeShortestEdge

ROOT = Path('grupo_N'); ROOT.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE = False  # True: comprueba el pipeline con pocos datos; NO genera resultados defendibles.
EPOCHS = 12 if not SMOKE else 1
BATCH = 2
WORKERS = 0
SEEDS = [42]  # Para resultados finales más robustos: [42, 123, 2026], antes de entrenar.
RUN_TTA = True
RUN_ENSEMBLE = True
ALLOW_TEST = False  # Cambiar solo después de revisar y bloquear validación.

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

def sync():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()

seed_all(SEEDS[0])
assert DEVICE.type == 'cuda' or SMOKE, 'Para el estudio completo utiliza GPU; CPU solo en smoke.'
env = {'python': sys.version, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
       'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU',
       'smoke': SMOKE, 'seeds': SEEDS}
(ROOT/'environment.json').write_text(json.dumps(env,indent=2))
(ROOT/'requirements_executed.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print(env)


## 1. Datos, etiquetas y auditoría
Se descarga el Parquet del repositorio fijando su revisión. Las cajas son COCO `[x,y,w,h]`. Se consultan los nombres e IDs en las anotaciones originales del mismo repositorio, sin asumir que sean índices 0–4: pueden incluir una categoría auxiliar sin instancias. Solo las cinco categorías presentes en train definen el problema y se verifica que las demás particiones no introduzcan otras.

Los IDs internos comunes son 0–4; SSD usa 1–5 y reserva 0 para fondo. Detectron2 usa 0–4 y añade su fondo internamente. Se recortan cajas al tamaño real y se registra cuántas quedan inválidas. Se preservan imágenes vacías. Se buscan duplicados exactos entre splits y se informa, sin redistribuir las particiones. Una ausencia de duplicados exactos no descarta imágenes casi idénticas.


In [ ]:
REPO = 'Francesco/lettuce-pallets'
revfile = ROOT/'dataset_revision.txt'
if not revfile.exists(): revfile.write_text(HfApi().dataset_info(REPO).sha)
REVISION = revfile.read_text().strip()
ds = load_dataset(REPO, revision=REVISION)
assert all(s in ds for s in ['train','validation','test'])
print(ds)
raw_ids = sorted({int(c) for obj in ds['train']['objects'] for c in obj['category']})
assert len(raw_ids) == 5, f'Revisar categorías: {raw_ids}'
archive = hf_hub_download(REPO, 'dataset.tar.gz', repo_type='dataset', revision=REVISION)
id_names = {}
with tarfile.open(archive) as tar:
    for member in tar.getmembers():
        if member.isfile() and member.name.endswith('.json'):
            try:
                obj = json.load(tar.extractfile(member))
            except (ValueError, UnicodeDecodeError):
                continue
            if isinstance(obj, dict):
                for c in obj.get('categories', []):
                    cid, name = int(c['id']), str(c['name'])
                    if cid in id_names: assert id_names[cid] == name
                    id_names[cid] = name
assert all(c in id_names for c in raw_ids), 'No se pudieron resolver nombres desde COCO original.'
RAW2IDX = {c:i for i,c in enumerate(raw_ids)}
NAMES = [id_names[c] for c in raw_ids]; K = len(NAMES)
(ROOT/'classes.json').write_text(json.dumps({'raw_to_internal':RAW2IDX, 'names':NAMES},indent=2))
print(dict(enumerate(NAMES)))

records = {}; audit = []; hashes = {}
for split in ['train','validation','test']:
    folder = ROOT/'data'/split; folder.mkdir(parents=True, exist_ok=True)
    recs=[]; ann=[]; imgs=[]; bad=0
    limit = min(len(ds[split]), 16 if split=='train' else 8) if SMOKE else len(ds[split])
    for i in tqdm(range(limit), desc=split):
        row=ds[split][i]; im=row['image'].convert('RGB'); w,h=im.size
        path=folder/f'{i:06d}.png'
        if not path.exists(): im.save(path)
        digest=hashlib.sha256(im.tobytes()+str(im.size).encode()).hexdigest()
        hashes.setdefault(digest,[]).append((split,i))
        boxes=[]; labels=[]
        for bbox, raw in zip(row['objects']['bbox'],row['objects']['category']):
            assert int(raw) in RAW2IDX, f'Categoría desconocida en {split}'
            x,y,bw,bh=map(float,bbox)
            a,b,c,d=np.clip([x,y,x+bw,y+bh],[0,0,0,0],[w,h,w,h])
            if not np.isfinite([a,b,c,d]).all() or c<=a or d<=b:
                bad+=1; continue
            label=RAW2IDX[int(raw)]
            boxes.append([a,b,c,d]); labels.append(label)
            ann.append({'id':len(ann)+1,'image_id':i,'category_id':label,
                        'bbox':[a,b,c-a,d-b],'area':(c-a)*(d-b),'iscrowd':0})
            audit.append({'split':split,'class':NAMES[label],'area':(c-a)*(d-b),
                          'relative_area':(c-a)*(d-b)/(w*h),'aspect':(c-a)/(d-b)})
        recs.append({'id':i,'path':str(path),'width':w,'height':h,'boxes':boxes,'labels':labels})
        imgs.append({'id':i,'width':w,'height':h,'file_name':str(path)})
    records[split]=recs
    coco={'info':{},'licenses':[],'images':imgs,'annotations':ann,
          'categories':[{'id':j,'name':n} for j,n in enumerate(NAMES)]}
    (ROOT/f'{split}.json').write_text(json.dumps(coco))
    print(split,'imágenes',len(recs),'cajas inválidas descartadas',bad,'vacías',sum(not r['boxes'] for r in recs))
(ROOT/'records.json').write_text(json.dumps(records))
dups=[v for v in hashes.values() if len({x[0] for x in v})>1]
(ROOT/'duplicates_cross_split.json').write_text(json.dumps(dups,indent=2))
print('Grupos de duplicados exactos entre splits:',len(dups))
audit=pd.DataFrame(audit); audit.to_csv(ROOT/'dataset_audit.csv',index=False)
# EDA únicamente de train/validation: no inspeccionar resultados del test antes del bloqueo.
eda=audit[audit.split!='test']
display(pd.crosstab(eda['class'],eda['split']))
fig,axes=plt.subplots(1,3,figsize=(16,4))
pd.crosstab(eda['class'],eda['split']).plot.bar(ax=axes[0])
axes[1].hist(np.sqrt(eda[eda.split=='train'].area),bins=35); axes[1].set_xlabel('Raíz del área (px originales)')
axes[2].hist(eda[eda.split=='train'].aspect,bins=35); axes[2].set_xlabel('Relación ancho/alto')
fig.tight_layout(); fig.savefig(ROOT/'eda.png',dpi=160); plt.show()


## 2. Transformaciones y modelos
El baseline no incluye aumentos aleatorios. `aug` añade flips horizontal/vertical, rotaciones de 90° y jitter suave. La transformación geométrica se aplica a las cuatro esquinas de las cajas mediante las mismas operaciones. Se incluye una comprobación sintética de los flips.

Cascade se construye con la configuración exacta del enunciado y `MASK_ON=False`. Sus salidas de clasificación se adaptan al número de clases; Detectron2 conserva los pesos compatibles al cargar COCO. Los avisos de dimensiones incompatibles de los clasificadores son esperables; errores del backbone no lo son. SSD conserva su regresor COCO y sustituye su clasificador.


In [ ]:
def augment(im, boxes):
    w,h=im.size
    if random.random()<.5:
        im=TF.hflip(im); boxes=boxes.clone(); boxes[:,[0,2]]=w-boxes[:,[2,0]]
    if random.random()<.5:
        im=TF.vflip(im); boxes=boxes.clone(); boxes[:,[1,3]]=h-boxes[:,[3,1]]
    for _ in range(random.randrange(4)):
        im=im.transpose(Image.Transpose.ROTATE_90)
        old=boxes.clone()
        boxes[:,0]=old[:,1]; boxes[:,2]=old[:,3]
        boxes[:,1]=w-old[:,2]; boxes[:,3]=w-old[:,0]
        w,h=h,w
    im=TF.adjust_brightness(im,random.uniform(.9,1.1))
    im=TF.adjust_contrast(im,random.uniform(.9,1.1))
    im=TF.adjust_saturation(im,random.uniform(.95,1.05))
    return im, boxes

b=torch.tensor([[10.,20.,40.,60.]])
b2=b.clone(); b2[:,[0,2]]=100-b2[:,[2,0]]
b2[:,[0,2]]=100-b2[:,[2,0]]
assert torch.equal(b,b2)

class Lettuce(Dataset):
    def __init__(self, split, aug=False): self.rows=records[split]; self.aug=aug
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]; im=Image.open(r['path']).convert('RGB')
        boxes=torch.tensor(r['boxes'],dtype=torch.float32).reshape(-1,4)
        if self.aug: im,boxes=augment(im,boxes)
        return TF.to_tensor(im), {'boxes':boxes,'labels':torch.tensor(r['labels'],dtype=torch.int64),
                                  'image_id':torch.tensor(r['id'])}

def loader(split, aug=False, seed=42):
    return DataLoader(Lettuce(split,aug), batch_size=BATCH if split=='train' else 1,
                      shuffle=split=='train', num_workers=WORKERS,
                      generator=torch.Generator().manual_seed(seed),
                      collate_fn=lambda batch: tuple(zip(*batch)))

CASCADE_CONFIG='Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml'
def make_model(arch, pretrained=True):
    if arch=='ssd':
        model=ssd300_vgg16(weights=SSD300_VGG16_Weights.COCO_V1 if pretrained else None,
                          weights_backbone=None,trainable_backbone_layers=5,
                          score_thresh=.001,nms_thresh=.45,detections_per_img=100)
        channels=[m.in_channels for m in model.head.classification_head.module_list]
        model.head.classification_head=SSDClassificationHead(channels,model.anchor_generator.num_anchors_per_location(),K+1)
    else:
        cfg=get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(CASCADE_CONFIG))
        cfg.MODEL.MASK_ON=False; cfg.MODEL.ROI_HEADS.NUM_CLASSES=K
        cfg.MODEL.BACKBONE.FREEZE_AT=0
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST=.001
        cfg.TEST.DETECTIONS_PER_IMAGE=100
        cfg.MODEL.DEVICE=str(DEVICE)
        model=build_model(cfg)
        if pretrained: DetectionCheckpointer(model).load(model_zoo.get_checkpoint_url(CASCADE_CONFIG))
    return model.to(DEVICE)

resize_d2=ResizeShortestEdge([640,640],1024)
def cascade_input(image,target=None):
    # Tensor RGB [0,1] -> BGR [0,255], entrada de la receta COCO de Detectron2.
    arr=(image.permute(1,2,0).cpu().numpy()*255).astype(np.float32)
    h,w=arr.shape[:2]; tr=resize_d2.get_transform(arr)
    transformed=tr.apply_image(arr)[:,:,::-1].copy()
    inp={'image':torch.as_tensor(transformed.transpose(2,0,1)).to(DEVICE),'height':h,'width':w}
    if target is not None:
        inst=Instances(transformed.shape[:2])
        boxes=tr.apply_box(target['boxes'].cpu().numpy())
        inst.gt_boxes=Boxes(torch.as_tensor(boxes,dtype=torch.float32,device=DEVICE))
        inst.gt_classes=target['labels'].to(DEVICE)
        inp['instances']=inst
    return inp

def training_losses(model,arch,images,targets):
    if arch=='cascade': return model([cascade_input(i,t) for i,t in zip(images,targets)])
    ts=[{'boxes':t['boxes'].to(DEVICE),'labels':t['labels'].to(DEVICE)+1} for t in targets]
    return model([i.to(DEVICE) for i in images],ts)

@torch.inference_mode()
def predict_one(model,arch,image):
    model.eval()
    if arch=='cascade':
        out=model([cascade_input(image)])[0]['instances'].to('cpu')
        return {'boxes':out.pred_boxes.tensor,'scores':out.scores,'labels':out.pred_classes}
    out=model([image.to(DEVICE)])[0]
    return {'boxes':out['boxes'].cpu(),'scores':out['scores'].cpu(),'labels':out['labels'].cpu()-1}

def coco_rows(pred,image_id):
    rows=[]
    for b,s,c in zip(pred['boxes'].tolist(),pred['scores'].tolist(),pred['labels'].tolist()):
        x,y,x2,y2=b
        rows.append({'image_id':int(image_id),'category_id':int(c),'score':float(s),
                     'bbox':[x,y,x2-x,y2-y]})
    return rows

def evaluate_rows(rows,split,save=None):
    gt=COCO(str(ROOT/f'{split}.json'))
    if rows: dt=gt.loadRes(rows)
    else:
        dt=COCO(); dt.dataset={'images':gt.dataset['images'],'categories':gt.dataset['categories'],'annotations':[]}; dt.createIndex()
    ev=COCOeval(gt,dt,'bbox'); ev.params.maxDets=[1,10,100]
    ev.evaluate(); ev.accumulate(); ev.summarize()
    keys=['AP','AP50','AP75','AP_small','AP_medium','AP_large','AR1','AR10','AR100','AR_small','AR_medium','AR_large']
    metrics={k:float(v) for k,v in zip(keys,ev.stats)}
    perclass={}
    for j,c in enumerate(ev.params.catIds):
        vals=ev.eval['precision'][:,:,j,0,-1]; vals=vals[vals>=0]
        perclass[NAMES[c]]=float(vals.mean()) if len(vals) else None
    metrics['per_class_AP']=perclass
    if save:
        save=Path(save); save.mkdir(parents=True,exist_ok=True)
        (save/'metrics.json').write_text(json.dumps(metrics,indent=2))
        (save/'predictions.json').write_text(json.dumps(rows))
        np.savez_compressed(save/'pr.npz',precision=ev.eval['precision'],recall=ev.params.recThrs)
    return metrics,ev

@torch.inference_mode()
def infer_split(fn,split):
    rows=[]
    for image,t in tqdm(Lettuce(split),desc=f'inferencia {split}'):
        rows.extend(coco_rows(fn(image),t['image_id']))
    return rows


## 3. Ablaciones de entrenamiento
Se mantiene el mismo presupuesto de épocas y batch por arquitectura. Las tasas base son específicas de arquitectura; no se afirma que sean óptimas. `diff_lr` reduce únicamente la tasa del backbone al 10%; `adamw` cambia el optimizador (incluyendo su regularización propia); `aug` cambia solo los aumentos. No se combinan mejoras antes de demostrar su efecto individual.

Se guarda el mejor checkpoint por AP de validación y uno de recuperación con optimizador, scheduler y RNG. Si el proceso se corta, se continúa la siguiente época desde `last.pt`. No interrumpir y reanudar con otros hiperparámetros. Con una semilla no puede estimarse varianza entre entrenamientos; cambiar SEEDS antes de empezar para obtenerla. El modo smoke comprueba conexiones y no permite test final.


In [ ]:
EXPERIMENTS=[
    dict(name='baseline',aug=False,optimizer='sgd',backbone_factor=1.),
    dict(name='aug',aug=True,optimizer='sgd',backbone_factor=1.),
    dict(name='adamw',aug=False,optimizer='adamw',backbone_factor=1.),
    dict(name='diff_lr',aug=False,optimizer='sgd',backbone_factor=.1),
]

def train_run(arch,exp,seed):
    seed_all(seed)
    out=ROOT/'runs'/f'{arch}_{exp["name"]}_s{seed}'; out.mkdir(parents=True,exist_ok=True)
    recipe={'arch':arch,'experiment':exp,'seed':seed,'epochs':EPOCHS,'batch':BATCH,
            'revision':REVISION,'smoke':SMOKE,'cascade_short_edge':640,'ssd_size':300}
    conf=out/'recipe.json'
    if conf.exists(): assert json.loads(conf.read_text())==recipe, 'Receta modificada: usa otro directorio.'
    conf.write_text(json.dumps(recipe,indent=2))
    if (out/'done.json').exists(): return json.loads((out/'done.json').read_text())
    model=make_model(arch)
    lr=.002 if arch=='cascade' else .001
    backbone=[]; head=[]
    for name,p in model.named_parameters():
        if p.requires_grad: (backbone if name.startswith('backbone.') else head).append(p)
    groups=[{'params':backbone,'lr':lr*exp['backbone_factor']},{'params':head,'lr':lr}]
    if exp['optimizer']=='sgd': opt=torch.optim.SGD(groups,momentum=.9,weight_decay=1e-4)
    else: opt=torch.optim.AdamW(groups,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    start=0; best=-1.; history=[]; total_time=0.
    if (out/'last.pt').exists():
        ck=torch.load(out/'last.pt',map_location='cpu',weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer']); sched.load_state_dict(ck['scheduler'])
        start=ck['epoch']+1; best=ck['best']; history=ck['history']; total_time=ck['train_seconds']
        random.setstate(ck['rng_python']); np.random.set_state(ck['rng_numpy']); torch.set_rng_state(ck['rng_torch'])
        if DEVICE.type=='cuda': torch.cuda.set_rng_state_all(ck['rng_cuda'])
    for epoch in range(start,EPOCHS):
        model.train(); losses_epoch=[]; sync(); begin=time.perf_counter()
        # Generator y aumentos reproducibles por época, incluso tras reanudar.
        seed_all(seed+epoch)
        for images,targets in tqdm(loader('train',exp['aug'],seed+epoch),desc=f'{out.name} época {epoch+1}'):
            losses=training_losses(model,arch,images,targets); loss=sum(losses.values())
            if not torch.isfinite(loss): raise RuntimeError(f'Pérdida no finita: {losses}')
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),10.); opt.step()
            losses_epoch.append(float(loss.detach()))
        sync(); seconds=time.perf_counter()-begin; total_time+=seconds; sched.step()
        rows=infer_split(lambda im:predict_one(model,arch,im),'validation')
        metrics,_=evaluate_rows(rows,'validation',out/f'epoch_{epoch+1:02d}')
        history.append({'epoch':epoch+1,'loss':np.mean(losses_epoch),'train_seconds':seconds,
                        'AP':metrics['AP'],'AP50':metrics['AP50']})
        if metrics['AP']>best:
            best=metrics['AP']; torch.save(model.state_dict(),out/'best.pt')
            (out/'best_metrics.json').write_text(json.dumps(metrics,indent=2))
        pd.DataFrame(history).to_csv(out/'history.csv',index=False)
        torch.save({'model':model.state_dict(),'optimizer':opt.state_dict(),'scheduler':sched.state_dict(),
                    'epoch':epoch,'best':best,'history':history,'train_seconds':total_time,
                    'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),
                    'rng_torch':torch.get_rng_state(),
                    'rng_cuda':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else []},out/'last.pt')
    result={'arch':arch,'experiment':exp['name'],'seed':seed,'AP':best,
            'AP50':json.loads((out/'best_metrics.json').read_text())['AP50'],
            'train_seconds':total_time,'run':str(out)}
    (out/'done.json').write_text(json.dumps(result,indent=2))
    del model,opt; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return result

results=[]
for seed in SEEDS:
    for arch in ['ssd','cascade']:
        for exp in EXPERIMENTS: results.append(train_run(arch,exp,seed))
results=pd.DataFrame(results); results.to_csv(ROOT/'validation_training.csv',index=False)
display(results)


## 4. Evolución y selección
La semilla primaria se fija a la primera de SEEDS, no a la que mejor salga. Si hay varias semillas, la receta ganadora se escoge por AP media de validación; se informa también la desviación estándar. Los tiempos de entrenamiento excluyen validación. Las pérdidas de Cascade y SSD no son directamente comparables: usar sus curvas para diagnosticar, no para decidir cuál detecta mejor.


In [ ]:
summary=results.groupby(['arch','experiment']).agg(AP_mean=('AP','mean'),AP_std=('AP','std'),AP50_mean=('AP50','mean')).reset_index()
summary.to_csv(ROOT/'ablation_summary.csv',index=False); display(summary)
chosen={}
for arch in ['ssd','cascade']:
    name=summary[summary.arch==arch].sort_values('AP_mean',ascending=False).iloc[0].experiment
    chosen[arch]=str(ROOT/'runs'/f'{arch}_{name}_s{SEEDS[0]}')
fig,axes=plt.subplots(1,2,figsize=(13,4))
for _,r in results[results.seed==SEEDS[0]].iterrows():
    hist=pd.read_csv(Path(r.run)/'history.csv')
    axes[0].plot(hist.epoch,hist.AP,label=f'{r.arch}/{r.experiment}')
    axes[1].plot(hist.epoch,hist.loss,label=f'{r.arch}/{r.experiment}')
axes[0].set_ylabel('AP validación'); axes[1].set_ylabel('Pérdida entrenamiento (escalas distintas)')
for ax in axes: ax.set_xlabel('Época'); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(ROOT/'learning_curves.png',dpi=170); plt.show()
print('Recetas seleccionadas:',chosen)


## 5. TTA y ensemble: hipótesis adicionales sin reentrenar
TTA combina la imagen original y su reflejo horizontal deshaciendo las coordenadas. El ensemble combina los dos ganadores de validación. Se usa NMS por clase con IoU=0.5 y como máximo 100 detecciones por imagen. No es WBF ni una calibración probabilística; las puntuaciones entre modelos pueden no ser comparables. No se promete que mejore. Estos parámetros quedan fijados antes de evaluar y todas las variantes se comparan en validación.

La latencia se mide con batch=1, calentamiento, sincronización CUDA y las mismas imágenes de validación. Incluye adaptación de entrada, transferencias, resize, forward y postprocesado; excluye leer la imagen del disco. Es una latencia de pipeline, no exclusivamente del kernel GPU. TTA y ensemble se cronometran realmente, no se estiman sumando tiempos.


In [ ]:
def load_run(path):
    recipe=json.loads((Path(path)/'recipe.json').read_text()); arch=recipe['arch']
    model=make_model(arch,pretrained=False)
    model.load_state_dict(torch.load(Path(path)/'best.pt',map_location=DEVICE,weights_only=True)); model.eval()
    return model,arch

def merge_predictions(preds,iou=.5):
    boxes=torch.cat([p['boxes'] for p in preds]); scores=torch.cat([p['scores'] for p in preds]); labels=torch.cat([p['labels'] for p in preds])
    keep=batched_nms(boxes,scores,labels,iou)[:100]
    return {'boxes':boxes[keep],'scores':scores[keep],'labels':labels[keep]}

def tta_predict(fn,im):
    original=fn(im); flipped=fn(im.flip(-1)); w=im.shape[-1]
    flipped['boxes']=flipped['boxes'].clone()
    flipped['boxes'][:,[0,2]]=w-flipped['boxes'][:,[2,0]]
    return merge_predictions([original,flipped])

models={}; functions={}; specs={}
# Mantener ambos modelos en GPU requiere memoria adicional. Si hay OOM, reducir BATCH
# no resuelve la residencia simultánea: desactivar ensemble o utilizar GPU con más memoria.
for arch,path in chosen.items():
    models[arch],_=load_run(path)
    functions[arch]=lambda im,a=arch:predict_one(models[a],a,im)
    specs[arch]={'type':'single','arch':arch,'run':path}
if RUN_TTA:
    for arch in ['ssd','cascade']:
        functions[arch+'_tta']=lambda im,a=arch:tta_predict(functions[a],im)
        specs[arch+'_tta']={'type':'tta','base':arch,'iou':.5}
if RUN_ENSEMBLE:
    functions['ensemble']=lambda im:merge_predictions([functions['ssd'](im),functions['cascade'](im)])
    specs['ensemble']={'type':'ensemble','bases':['ssd','cascade'],'iou':.5}

@torch.inference_mode()
def benchmark(fn,n=50):
    images=[Lettuce('validation')[i][0] for i in range(min(n,len(records['validation'])))]
    for _ in range(5): fn(images[0])
    ms=[]
    for im in images:
        sync(); t=time.perf_counter(); fn(im); sync(); ms.append((time.perf_counter()-t)*1000)
    return {'latency_ms_median':float(np.median(ms)),'latency_ms_p90':float(np.percentile(ms,90)),
            'latency_ms_mean':float(np.mean(ms)),'timed_images':len(ms)}

variant_rows=[]
for name,fn in functions.items():
    pred=infer_split(fn,'validation')
    metrics,_=evaluate_rows(pred,'validation',ROOT/'validation_variants'/name)
    variant_rows.append({'variant':name,**{k:v for k,v in metrics.items() if k!='per_class_AP'},**benchmark(fn)})
variants=pd.DataFrame(variant_rows); variants.to_csv(ROOT/'validation_variants.csv',index=False)
display(variants[['variant','AP','AP50','AP75','latency_ms_median','latency_ms_p90']])


## 6. Curvas PR y análisis cualitativo en validación
Curvas COCO interpoladas por clase a IoU=.50, área total y maxDets=100. No confundir una curva PR con un umbral de confianza. El AP se calcula con el umbral interno bajo (.001), no filtrando previamente a .5.

Para visualizar errores se usa confianza .30 e IoU .50 fijados: matching voraz uno-a-uno dentro de cada clase, ordenado por confianza. Verde=TP, rojo=FP, naranja=FN. Los FP pueden deberse a mala clase, localización o duplicados; distinguirlos inspeccionando imagen y anotación. El recuento es diagnóstico, no sustituye la evaluación COCO. Se guardan ejemplos de más y menos errores; es una selección deliberada, no una muestra representativa.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,arch in zip(axes,['ssd','cascade']):
    pr=np.load(ROOT/'validation_variants'/arch/'pr.npz')
    for j,name in enumerate(NAMES):
        values=pr['precision'][0,:,j,0,-1]; ok=values>=0
        ax.plot(pr['recall'][ok],values[ok],label=name)
    ax.set(title=arch,xlabel='Recall',ylabel='Precision',xlim=(0,1),ylim=(0,1.02)); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(ROOT/'pr_validation.png',dpi=170); plt.show()

def diagnose(pred,target,threshold=.30):
    keep=pred['scores']>=threshold
    p={k:v[keep] for k,v in pred.items()}
    used=set(); tp=[]; fp=[]
    ious=box_iou(p['boxes'],target['boxes'])
    for j in p['scores'].argsort(descending=True).tolist():
        options=[k for k in range(len(target['labels'])) if k not in used and target['labels'][k]==p['labels'][j]]
        k=max(options,key=lambda k:float(ious[j,k])) if options else None
        if k is not None and ious[j,k]>=.5: tp.append(j); used.add(k)
        else: fp.append(j)
    fn=[k for k in range(len(target['labels'])) if k not in used]
    return p,tp,fp,fn

def draw_box(ax,b,label,color):
    x,y,x2,y2=map(float,b)
    ax.add_patch(plt.Rectangle((x,y),x2-x,y2-y,fill=False,color=color,linewidth=1.5))
    ax.text(x,y,label,color='black',fontsize=6,bbox=dict(facecolor=color,alpha=.75,pad=1))

def qualitative(fn,split,name):
    dsvis=Lettuce(split); ranked=[]
    for i in range(len(dsvis)):
        im,t=dsvis[i]; p,tp,fp,fnids=diagnose(fn(im),t)
        ranked.append({'index':i,'TP':len(tp),'FP':len(fp),'FN':len(fnids),'errors':len(fp)+len(fnids)})
    table=pd.DataFrame(ranked).sort_values('errors',ascending=False)
    dest=ROOT/'qualitative'/split/name; dest.mkdir(parents=True,exist_ok=True)
    table.to_csv(dest/'error_counts.csv',index=False)
    ids=list(dict.fromkeys(table.head(2)['index'].tolist()+table.tail(2)['index'].tolist()))
    for i in ids:
        im,t=dsvis[i]; p,tp,fp,fnids=diagnose(fn(im),t)
        fig,ax=plt.subplots(figsize=(9,9)); ax.imshow(im.permute(1,2,0))
        for j in tp+fp: draw_box(ax,p['boxes'][j],f"{NAMES[int(p['labels'][j])]} {float(p['scores'][j]):.2f}",'lime' if j in tp else 'red')
        for j in fnids: draw_box(ax,t['boxes'][j],'FN '+NAMES[int(t['labels'][j])],'orange')
        ax.set_title(f'{name} / {split} / {i}: TP={len(tp)} FP={len(fp)} FN={len(fnids)}'); ax.axis('off')
        fig.savefig(dest/f'{i:04d}.png',dpi=150,bbox_inches='tight'); plt.close(fig)
    return table

for arch in ['ssd','cascade']: qualitative(functions[arch],'validation',arch)


## 7. Bloqueo y evaluación final
Primero revisar validación. Esta celda crea un manifiesto inmutable con recetas, hashes de pesos y ganador. La comparación final incluye los dos baselines, el mejor entrenamiento de cada arquitectura y las variantes de inferencia predeclaradas. El ganador global queda elegido por validación. No volver a elegirlo según test.

Activar `ALLOW_TEST=True` en la configuración y ejecutar esta sección solo al cerrar la experimentación. Si se añaden experimentos después de ver test, declararlo como limitación: deja de ser una estimación final independiente. No interpretar una diferencia pequeña como significativa con una sola semilla.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

if ALLOW_TEST:
    assert not SMOKE, 'Test final prohibido en smoke.'
    final_specs=copy.deepcopy(specs)
    for arch in ['ssd','cascade']:
        path=str(ROOT/'runs'/f'{arch}_baseline_s{SEEDS[0]}')
        final_specs[arch+'_baseline']={'type':'single','arch':arch,'run':path}
    weight_paths=sorted({str(Path(s['run'])/'best.pt') for s in final_specs.values() if 'run' in s})
    manifest={'dataset_revision':REVISION,'winner_validation':variants.sort_values('AP',ascending=False).iloc[0]['variant'],
              'specs':final_specs,'weights':{p:sha256_file(p) for p in weight_paths},
              'seed_policy':'primera semilla prefijada; receta por media de AP validación',
              'evaluation':{'maxDets':[1,10,100],'score_floor':.001,'merge_iou':.5}}
    lock=ROOT/'LOCKED_TEST_PROTOCOL.json'
    if lock.exists(): assert json.loads(lock.read_text())==manifest, 'El protocolo ya bloqueado ha cambiado.'
    else: lock.write_text(json.dumps(manifest,indent=2))
    final=[]
    for name,spec in final_specs.items():
        if name in functions: fn=functions[name]; temporary=None
        else:
            temporary,arch=load_run(spec['run'])
            fn=lambda im,m=temporary,a=arch:predict_one(m,a,im)
        rows=infer_split(fn,'test'); met,_=evaluate_rows(rows,'test',ROOT/'test_results'/name)
        timing=benchmark(fn)
        final.append({'variant':name,**{k:v for k,v in met.items() if k!='per_class_AP'},**timing})
        if name==manifest['winner_validation']: qualitative(fn,'test',name)
        if temporary is not None:
            del fn,temporary; gc.collect()
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
    final=pd.DataFrame(final); final.to_csv(ROOT/'test_comparison.csv',index=False)
    display(final[['variant','AP','AP50','AP75','AP_small','AP_medium','AP_large','latency_ms_median']])
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    final.set_index('variant')[['AP','AP50']].plot.bar(ax=axes[0]); axes[0].set_ylim(0,1)
    axes[1].scatter(final.latency_ms_median,final.AP)
    for _,r in final.iterrows(): axes[1].annotate(r.variant,(r.latency_ms_median,r.AP),fontsize=7)
    axes[1].set(xlabel='Latencia mediana batch=1 (ms), medida en validación',ylabel='AP test')
    fig.tight_layout(); fig.savefig(ROOT/'final_comparison.png',dpi=180); plt.show()
else:
    print('Test no ejecutado: revisa validación y activa ALLOW_TEST cuando el protocolo sea definitivo.')


## 8. Interpretación que debéis completar con resultados reales

- ¿Qué detector tiene mayor AP? ¿Coincide el orden en AP50 y AP75? Una ganancia solo en AP50 no demuestra mejor localización estricta.
- ¿Qué clases fallan más? Cruzar AP por clase, frecuencia, tamaño y ejemplos; la correlación no prueba causalidad.
- ¿Los flips/rotaciones ayudan sin destruir información relevante? Comprobar la hipótesis de invariancia cenital.
- ¿AdamW aporta más que SGD con este presupuesto? No generalizar fuera de las tasas y épocas probadas.
- ¿La tasa diferenciada preserva mejor los rasgos COCO o limita la adaptación al dominio agrícola?
- ¿TTA/ensemble justifican su coste? Presentar Pareto precisión/latencia, no solo el máximo AP.
- Si se usaron varias semillas: mostrar media ± desviación y declarar que el test de esta implementación usa la semilla primaria prefijada. Para estimar varianza de test haría falta predeclarar y evaluar todas las semillas, sin seleccionar la mejor.
- AP=-1 significa que esa categoría de tamaño no es evaluable; no es una precisión negativa.
- Limitaciones: particiones pequeñas, posibles duplicados cercanos, versiones de paquetes, presupuesto y falta de búsqueda exhaustiva. Las semillas ayudan, pero kernels/entornos distintos pueden impedir igualdad bit a bit.

No escribir «la mejora funciona» si el cambio es negativo. Una hipótesis rechazada con evidencia y análisis es un resultado experimental válido.

## Defensa propuesta: 11–12 minutos + preguntas
1. Integrante 1 (≈4 min): dataset, auditoría, arquitecturas y protocolo.
2. Integrante 2 (≈4 min): baseline, hipótesis, ablaciones y curvas.
3. Integrante 3 (≈4 min): test bloqueado, PR, errores, velocidad y conclusiones.

Preparad respuestas sobre COCO xywh vs xyxy; fondo en SSD/Detectron2; AP50 vs AP@[.5:.95]; NMS; fuga de información; por qué se conserva SSD300; equidad de presupuesto; y por qué el ensemble no tiene por qué superar a cada modelo.

## Entrega
El ZIP requerido debe incluir notebook limpio/comentado, README y diapositivas PDF. Este notebook produce material para las diapositivas, pero no sustituye la presentación. No se inventan resultados ni se genera automáticamente un PDF de conclusiones antes de entrenar. Entrega un solo integrante. El grupo N no identifica por sí solo el grupo interactivo: confirmar si aplica el cierre del grupo 1 (30/11 11:30) o grupo 2 (26/11 16:30), aunque la defensa sea el día 30.

## Fuentes técnicas
- Enunciado facilitado, apartados 1–7.
- https://huggingface.co/datasets/Francesco/lettuce-pallets
- https://docs.pytorch.org/vision/main/_modules/torchvision/models/detection/ssd.html
- https://github.com/facebookresearch/detectron2/blob/main/configs/Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml
- https://detectron2.readthedocs.io/en/latest/tutorials/install.html
